# Test 7: Temporal Chaos Divergence

**Гипотеза:** Feedforward автоэнкодер не сохраняет экспоненциальную дивергенцию траекторий хаотических систем.

**План:**
- Генерация пар траекторий с близкими начальными условиями (x0, x0+delta)
- Кодирование каждого кадра в латентное пространство
- Измерение ||z1(t) - z2(t)|| для каждого шага
- Сравнение: True chaos baseline, Dense ReLU 64, Dense ReLU 128, V4 K-Sparse
- N=5 запусков с разными начальными условиями

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
from datetime import datetime

print(f"TF version: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

/Users/savenkovviktor/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TF version: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
def logistic_map(x, r=3.99):
    return r * x * (1 - x)


def generate_logistic_image(initial_value, image_size=28, r=3.99):
    iterations = image_size * image_size
    x = initial_value
    sequence = []
    for _ in range(iterations):
        x = logistic_map(x, r)
        sequence.append(x)
    img = np.array(sequence).reshape((image_size, image_size))
    return img


def generate_trajectory_pair(num_steps=10, initial_x0=0.4, delta=1e-6,
                             image_size=28, r=3.99):
    trajectory1 = []
    trajectory2 = []
    true_distances = []

    x1 = initial_x0
    x2 = initial_x0 + delta

    for step in range(num_steps):
        img1 = generate_logistic_image(x1, image_size, r)
        img2 = generate_logistic_image(x2, image_size, r)

        trajectory1.append(img1)
        trajectory2.append(img2)
        true_distances.append(abs(x1 - x2))

        x1 = logistic_map(x1, r)
        x2 = logistic_map(x2, r)

    trajectory1 = np.array(trajectory1)[..., np.newaxis].astype('float32')
    trajectory2 = np.array(trajectory2)[..., np.newaxis].astype('float32')

    return trajectory1, trajectory2, np.array(true_distances)


def generate_training_data(num_images=2000, image_size=28, r=3.99):
    dataset = []
    for _ in range(num_images):
        x0 = np.random.rand()
        img = generate_logistic_image(x0, image_size, r)
        dataset.append(img)
    return np.array(dataset)[..., np.newaxis].astype('float32')

In [3]:
def build_dense_relu_ae(latent_dim=64, image_size=28):
    input_img = keras.Input(shape=(image_size, image_size, 1))
    x = layers.Flatten()(input_img)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dense(128, activation='relu')(x)
    latent = layers.Dense(latent_dim, activation='relu', name='latent')(x)

    encoder = keras.Model(input_img, latent, name='encoder')

    x = layers.Dense(128, activation='relu')(latent)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dense(image_size * image_size, activation='sigmoid')(x)
    decoded = layers.Reshape((image_size, image_size, 1))(x)

    autoencoder = keras.Model(input_img, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')

    return autoencoder, encoder


def chaos_activation(x):
    return tf.sin(8.0 * x) + 0.5 * tf.tanh(4.0 * x)


class KSparseLayer(layers.Layer):
    def __init__(self, k=32, **kwargs):
        super().__init__(**kwargs)
        self.k = k

    def call(self, inputs):
        abs_inputs = tf.abs(inputs)
        _, indices = tf.nn.top_k(abs_inputs, k=self.k, sorted=False)
        mask = tf.reduce_sum(
            tf.one_hot(indices, depth=tf.shape(inputs)[-1], dtype=inputs.dtype),
            axis=1
        )
        return inputs * mask

    def get_config(self):
        config = super().get_config()
        config.update({'k': self.k})
        return config


def build_ksparse_chaos_ae(latent_dim=128, k=32, image_size=28):
    input_img = keras.Input(shape=(image_size, image_size, 1))
    x = layers.Flatten()(input_img)

    x = layers.Dense(256)(x)
    x = layers.Activation(chaos_activation)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    latent_raw = layers.Dense(latent_dim, name='latent_pre')(x)
    latent_activated = layers.Activation(chaos_activation)(latent_raw)
    latent = KSparseLayer(k=k, name='latent_sparse')(latent_activated)

    encoder = keras.Model(input_img, latent, name='encoder')

    x = layers.Dense(256)(latent)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(chaos_activation)(x)
    x = layers.Dropout(0.1)(x)

    x = layers.Dense(image_size * image_size, activation='sigmoid')(x)
    decoded = layers.Reshape((image_size, image_size, 1))(x)

    autoencoder = keras.Model(input_img, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')

    return autoencoder, encoder

In [4]:
def test_chaos_divergence(encoder, num_steps=10, num_runs=5,
                          initial_x0=0.4, delta=1e-6, image_size=28):
    all_ratios = []
    all_initial_distances = []
    all_final_distances = []

    for run in range(num_runs):
        x0 = initial_x0 + np.random.uniform(-0.1, 0.1)

        traj1, traj2, true_dist = generate_trajectory_pair(
            num_steps=num_steps,
            initial_x0=x0,
            delta=delta,
            image_size=image_size
        )

        latent1 = encoder.predict(traj1, verbose=0)
        latent2 = encoder.predict(traj2, verbose=0)

        latent_distances = [
            float(np.linalg.norm(latent1[i] - latent2[i]))
            for i in range(num_steps)
        ]

        if latent_distances[0] > 1e-10:
            ratio = latent_distances[-1] / latent_distances[0]
        else:
            ratio = np.nan

        all_ratios.append(ratio)
        all_initial_distances.append(latent_distances[0])
        all_final_distances.append(latent_distances[-1])

    valid_ratios = [r for r in all_ratios if not np.isnan(r)]

    if len(valid_ratios) == 0:
        return {
            'mean_ratio': np.nan,
            'std_ratio': np.nan,
            'mean_initial_distance': np.nan,
            'mean_final_distance': np.nan,
            'all_ratios': []
        }

    return {
        'mean_ratio': float(np.mean(valid_ratios)),
        'std_ratio': float(np.std(valid_ratios)),
        'mean_initial_distance': float(np.mean(all_initial_distances)),
        'std_initial_distance': float(np.std(all_initial_distances)),
        'mean_final_distance': float(np.mean(all_final_distances)),
        'std_final_distance': float(np.std(all_final_distances)),
        'all_ratios': valid_ratios,
        'num_valid_runs': len(valid_ratios)
    }


def compute_true_chaos_divergence(num_steps=10, num_runs=5,
                                  initial_x0=0.4, delta=1e-6, r=3.99):
    all_ratios = []

    for run in range(num_runs):
        x0 = initial_x0 + np.random.uniform(-0.1, 0.1)

        distances = []
        x1 = x0
        x2 = x0 + delta

        for step in range(num_steps):
            distances.append(abs(x1 - x2))
            x1 = logistic_map(x1, r)
            x2 = logistic_map(x2, r)

        if distances[0] > 1e-10:
            ratio = distances[-1] / distances[0]
            all_ratios.append(ratio)

    return {
        'mean_ratio': float(np.mean(all_ratios)),
        'std_ratio': float(np.std(all_ratios)),
        'all_ratios': all_ratios,
        'expected': '>5.0 for chaotic divergence (lambda ~ 0.5)'
    }

In [5]:
NUM_STEPS = 10
NUM_RUNS = 5
IMAGE_SIZE = 28
DELTA = 1e-6
EPOCHS = 10

results = {}

In [6]:
true_chaos = compute_true_chaos_divergence(
    num_steps=NUM_STEPS,
    num_runs=NUM_RUNS,
    delta=DELTA
)

print(f"True logistic map (scalar state):")
print(f"  Mean ratio: {true_chaos['mean_ratio']:.2f} +/- {true_chaos['std_ratio']:.2f}")
print(f"  Expected: {true_chaos['expected']}")

results['true_chaos'] = true_chaos

True logistic map (scalar state):
  Mean ratio: 236.37 +/- 185.76
  Expected: >5.0 for chaotic divergence (lambda ~ 0.5)


In [7]:
train_data = generate_training_data(num_images=2000, image_size=IMAGE_SIZE)
print(f"Train: {train_data.shape}")

Train: (2000, 28, 28, 1)


In [8]:
ae_dense64, enc_dense64 = build_dense_relu_ae(latent_dim=64, image_size=IMAGE_SIZE)
ae_dense64.fit(
    train_data, train_data,
    epochs=EPOCHS,
    batch_size=64,
    validation_split=0.1,
    verbose=0
)

div_dense64 = test_chaos_divergence(
    enc_dense64,
    num_steps=NUM_STEPS,
    num_runs=NUM_RUNS,
    delta=DELTA,
    image_size=IMAGE_SIZE
)

print(f"Dense ReLU 64:")
print(f"  Ratio: {div_dense64['mean_ratio']:.3f} +/- {div_dense64['std_ratio']:.3f}")
print(f"  Initial distance: {div_dense64['mean_initial_distance']:.2f} +/- {div_dense64['std_initial_distance']:.2f}")
print(f"  Final distance: {div_dense64['mean_final_distance']:.2f} +/- {div_dense64['std_final_distance']:.2f}")

results['dense_relu_64'] = div_dense64

2026-03-09 14:19:43.409064: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2026-03-09 14:19:43.409084: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 48.00 GB
2026-03-09 14:19:43.409087: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 18.00 GB
2026-03-09 14:19:43.409100: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-09 14:19:43.409108: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-03-09 14:19:44.040369: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Dense ReLU 64:
  Ratio: 1.011 +/- 0.270
  Initial distance: 6.87 +/- 0.83
  Final distance: 6.74 +/- 1.24


In [9]:
ae_dense128, enc_dense128 = build_dense_relu_ae(latent_dim=128, image_size=IMAGE_SIZE)
ae_dense128.fit(
    train_data, train_data,
    epochs=EPOCHS,
    batch_size=64,
    validation_split=0.1,
    verbose=0
)

div_dense128 = test_chaos_divergence(
    enc_dense128,
    num_steps=NUM_STEPS,
    num_runs=NUM_RUNS,
    delta=DELTA,
    image_size=IMAGE_SIZE
)

print(f"Dense ReLU 128:")
print(f"  Ratio: {div_dense128['mean_ratio']:.3f} +/- {div_dense128['std_ratio']:.3f}")
print(f"  Initial distance: {div_dense128['mean_initial_distance']:.2f} +/- {div_dense128['std_initial_distance']:.2f}")
print(f"  Final distance: {div_dense128['mean_final_distance']:.2f} +/- {div_dense128['std_final_distance']:.2f}")

results['dense_relu_128'] = div_dense128

Dense ReLU 128:
  Ratio: 1.047 +/- 0.140
  Initial distance: 7.51 +/- 1.03
  Final distance: 7.73 +/- 0.23


In [10]:
ae_v4, enc_v4 = build_ksparse_chaos_ae(latent_dim=128, k=32, image_size=IMAGE_SIZE)
ae_v4.fit(
    train_data, train_data,
    epochs=EPOCHS,
    batch_size=64,
    validation_split=0.1,
    verbose=0
)

div_v4 = test_chaos_divergence(
    enc_v4,
    num_steps=NUM_STEPS,
    num_runs=NUM_RUNS,
    delta=DELTA,
    image_size=IMAGE_SIZE
)

print(f"V4 K-Sparse Chaos:")
print(f"  Ratio: {div_v4['mean_ratio']:.3f} +/- {div_v4['std_ratio']:.3f}")
print(f"  Initial distance: {div_v4['mean_initial_distance']:.2f} +/- {div_v4['std_initial_distance']:.2f}")
print(f"  Final distance: {div_v4['mean_final_distance']:.2f} +/- {div_v4['std_final_distance']:.2f}")

results['v4_ksparse'] = div_v4

V4 K-Sparse Chaos:
  Ratio: 0.951 +/- 0.079
  Initial distance: 11.21 +/- 0.67
  Final distance: 10.61 +/- 0.26


In [11]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"temporal_chaos_divergence_results_{timestamp}.json"

with open(filename, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {filename}")

Results saved to: temporal_chaos_divergence_results_20260309_142002.json
